# Clase 08 · Outliers multivariados con algoritmos de ML (práctica)

> El video se titula *"Outliers en R"*, pero todo es **Python**: es la continuación de la clase 07.

En la clase 07 eliminamos outliers **columna por columna** (IQR) y perdimos casi la mitad del dataset.
El profesor propone otra idea: **tratar la detección de outliers como una clasificación binaria** (inlier / outlier)
hecha por un algoritmo que mira **todas las columnas a la vez**.

El profesor usó la librería **PyOD**. Aquí usamos sus equivalentes de **scikit-learn** (ya instalado):

| PyOD (clase) | scikit-learn (esta práctica) |
|---|---|
| `KNN(method="largest")` | `NearestNeighbors` + distancia al k-ésimo vecino |
| `IForest` | `IsolationForest` |
| `LOF` | `LocalOutlierFactor` |
| etiquetas `0` = inlier, `1` = outlier | etiquetas `1` = inlier, `-1` = outlier |

Dataset: `data/raw/viviendas_lima.csv`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.ensemble import IsolationForest

# Busca la raíz del repo (la carpeta que contiene data/raw)
raiz = Path.cwd()
while not (raiz / "data" / "raw").exists() and raiz != raiz.parent:
    raiz = raiz.parent
df = pd.read_csv(raiz / "data" / "raw" / "viviendas_lima.csv")
df.shape

## 1. Cada fila es un punto en R^n

Como el profesor quitó la columna objetivo (`MEDV` en Boston → 506 × 13), aquí quitamos `precio_usd`
(lo que queremos predecir) y nos quedamos con 6 columnas numéricas. Cada vivienda es entonces **un punto en dimensión 6**.

Para que el ejemplo sea simple, usamos solo filas sin NaN (en la vida real imputarías primero, como en la clase 03).

In [ ]:
cols = ["area_m2", "area_construida_m2", "habitaciones", "banos", "antiguedad_anios", "distancia_centro_km"]
df_tmp = df.dropna(subset=cols).copy()   # copia, como hizo el profesor
X = df_tmp[cols]
print(X.shape)        # (filas, dimensión del espacio)
X.iloc[0].to_numpy()  # la primera vivienda: un vector de 6 números = un punto

## 2. kNN: los puntos aislados son outliers

Idea: para cada punto miramos sus `k` vecinos más cercanos. Si están **lejos**, el punto está aislado → candidato a outlier.

Tres formas de resumir esas distancias (las mismas de PyOD):
- `largest`: distancia al k-ésimo vecino
- `mean`: promedio de las k distancias
- `median`: mediana de las k distancias

⚠️ Trampa: si buscas los vecinos de los mismos puntos con los que entrenaste, **el vecino 0 es el propio punto** (distancia 0).
Por eso pedimos `k + 1` y descartamos la primera columna. Veámoslo con 5 números:

In [ ]:
mini = pd.DataFrame({"x": [0, 1, 2, 3, 10]})
nn = NearestNeighbors(n_neighbors=3 + 1).fit(mini)
dist, idx = nn.kneighbors(mini)
print("distancias (con el propio punto en la columna 0):\n", dist)
print("índices de los vecinos:\n", idx)

In [ ]:
d = dist[:, 1:]          # quitamos el propio punto
print("largest:", d[:, -1])
print("mean:   ", d.mean(axis=1).round(2))
print("median: ", np.median(d, axis=1))
# El 10 tiene el puntaje más alto en los tres métodos -> es el outlier

Ahora sobre las viviendas (k = 5):

In [ ]:
k = 5
nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
dist, _ = nn.kneighbors(X)
puntaje = dist[:, -1]                    # método "largest"
df_tmp["puntaje_knn"] = puntaje
df_tmp.sort_values("puntaje_knn", ascending=False)[cols + ["puntaje_knn"]].head()

Las más aisladas son las viviendas con `area_m2` gigante (los errores ×10 de la clase 06).

## 3. `contamination`: cuánto dataset aceptas sacrificar

El profesor: con `contamination=0.2` le dices *"compromete a lo mucho el 20 % de la base de datos, y elimina los más alejados"*.
PyOD lo hace así: calcula el percentil `(1 - contamination)` de los puntajes y marca como outlier (1) a los que lo superan.

In [ ]:
contamination = 0.2
umbral = np.percentile(puntaje, 100 * (1 - contamination))
pred = (puntaje > umbral).astype(int)        # 0 = inlier, 1 = outlier (convención PyOD)
print("umbral:", round(umbral, 2))
print("inliers:", (pred == 0).sum(), " outliers:", (pred == 1).sum())

Compara con la clase 07: con IQR columna por columna se perdía cerca de la mitad de las filas.
Aquí **tú decides el tope** (20 %) y se eliminan los puntos más alejados en el espacio completo.

### ✏️ Tu turno 1
Repite el cálculo con el método `"mean"` (promedio de las 5 distancias) y `contamination=0.1`.
¿Cuántos outliers salen? ¿Sigue apareciendo la vivienda de 2746 m² entre ellos?

In [ ]:
# Tu código aquí


## 4. Otro algoritmo: IsolationForest (y la traducción de etiquetas)

scikit-learn devuelve `1` = inlier y `-1` = outlier. Para comparar con PyOD traducimos a `0` / `1`.

In [ ]:
iso = IsolationForest(contamination=0.2, random_state=42)
pred_sk = iso.fit_predict(X)                  # 1 / -1
print(pd.Series(pred_sk).value_counts())
pred_iso = np.where(pred_sk == -1, 1, 0)      # -> 0 / 1 como PyOD
print("outliers IsolationForest:", pred_iso.sum())

## 5. Cada algoritmo marca puntos distintos

En la clase: ABOD marcó 19 puntos y kNN 11 con la misma contaminación; CBLOF marcó como outliers una región que otros veían como inliers.
Midámoslo con la **intersección** y el **índice de Jaccard** (intersección / unión).

In [ ]:
def jaccard(a, b):
    a, b = np.asarray(a) == 1, np.asarray(b) == 1
    union = (a | b).sum()
    return 1.0 if union == 0 else (a & b).sum() / union

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.2).fit_predict(X)
pred_lof = np.where(lof == -1, 1, 0)

print("kNN vs IsolationForest -> coinciden:", (pred & pred_iso).sum(), " Jaccard:", round(jaccard(pred, pred_iso), 3))
print("kNN vs LOF             -> coinciden:", (pred & pred_lof).sum(), " Jaccard:", round(jaccard(pred, pred_lof), 3))

Los tres marcan ~325 viviendas, pero **no las mismas**. ¿Cuál es mejor? El profesor:
*"aquí no podemos saber nada"* mirando la gráfica. Se decide por el **rendimiento del modelo final** (protocolo de experimentación).

### ✏️ Tu turno 2
Escala las columnas antes de kNN (`from sklearn.preprocessing import StandardScaler`) y vuelve a calcular los outliers con contamination=0.2.
¿Cuánto se parecen (Jaccard) a los de kNN sin escalar? ¿Por qué crees que cambian? (Pista: mira en qué unidades está `area_m2` y en cuáles `habitaciones`.)

In [ ]:
# Tu código aquí


## 6. Quedarse con los inliers

Como el profesor (`df_tmp[pred == 0]` → 417 de 506):

In [ ]:
df_limpio = df_tmp[pred == 0].copy()
print(df_limpio.shape, f"-> conservado: {100 * len(df_limpio) / len(df_tmp):.2f} %")

### ✏️ Tu turno 3 (conexión con ML)
Ajusta una recta `precio_usd ≈ a · area_m2 + b` con `np.polyfit` usando `df_tmp` (todo) y `df_limpio` (solo inliers kNN).
¿Cómo cambia la pendiente `a`? Recuerda la gráfica de la clase 06: los outliers *"jalan"* la recta.

In [ ]:
# Tu código aquí


## 7. (Opcional) El código original con PyOD

Si algún día instalas PyOD (`pip install pyod`), esta celda reproduce lo de la clase. Si no está instalado, solo muestra un aviso.

In [ ]:
try:
    from pyod.models.knn import KNN
    clf = KNN(contamination=0.2)        # method="largest" por defecto
    clf.fit(X)
    pred_pyod = clf.predict(X)          # 0 = inlier, 1 = outlier
    print("inliers:", (pred_pyod == 0).sum(), " outliers:", (pred_pyod == 1).sum())
except ImportError:
    print("PyOD no está instalado: usa los equivalentes de scikit-learn de arriba.")